## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

1. **Critical Care Protocols:** "What is the protocol for managing sepsis in a critical care unit?"

2. **General Surgery:** "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

3. **Dermatology:** "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

4. **Neurology:** "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"


### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [27]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30 \
              langchain-huggingface \
              sentence-transformers \
              huggingface-hub \
              transformers \
              faiss-cpu \
              numpy==1.26.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requir

In [28]:
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 \
pip install -q llama-cpp-python==0.2.45 \
--force-reinstall --upgrade --no-cache-dir

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 120.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 187.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 324.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 167.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 271.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.86 which is incompatible.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 

In [16]:
import os

from google.colab import userdata

# PDF Loader
from langchain_community.document_loaders import PyPDFLoader

# Text Splitter (NEW IMPORT)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# Vector Store
from langchain_community.vectorstores import FAISS

## **LLM with Prompt Engineering Response**

#### **Download LLaMA-2 13B Chat Model**

In [1]:
from huggingface_hub import hf_hub_download

model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf"

model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

#### **Initialize LLaMA Model with Configuration**

In [2]:
from llama_cpp import Llama

lcpp_llm = Llama(
    model_path=model_path, # Path to the downloaded GGUF model
    n_threads=4,           # Number of CPU threads to use
    n_batch=512,           # Batch size for prompt processing
    n_gpu_layers=-1,       # Number of layers to offload to GPU (-1 for all)
    n_ctx=4096             # Context window
)

llama_model_loader: loaded meta data with 19 key-value pairs and 363 tensors from /root/.cache/huggingface/hub/models--TheBloke--Llama-2-13B-chat-GGUF/snapshots/4458acc949de0a9914c3eab623904d4fe999050a/llama-2-13b-chat.Q5_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 5120
llama_model_loader: - kv   4:                          llama.block_count u32              = 40
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 13824
llama_model_loader: - kv   6:                 llama.rope.dimension_

In [3]:
# Provides an example and answer anchor to guide the model in giving concise, evidence-based responses without echoing the question.
system_prompt = """
You are a medical expert AI assistant. Respond with concise, evidence-based answers.
"""

user_prompt = """
Example:
Q: What are the common symptoms of appendicitis?
A: Common symptoms include abdominal pain (usually starting near the navel), nausea, vomiting, and fever.
References: Mayo Clinic, UpToDate

Now answer:
Q: What is the protocol for managing sepsis in a critical care unit?
A:
"""


#### **Response Function**

In [4]:
#function to generate, process, and return the response from the LLM
def prompt_engineering_response(user_prompt):
    # Put the system message first, then the user question, then anchor with "Answer:"
    prompt = f"""{system_prompt}

Question: {user_prompt}

Answer:"""

    # Generate a response from the LLaMA model
    response = lcpp_llm(
        prompt=prompt,
        max_tokens=500, # Max number of tokens to generate
        temperature=0.3, # Sampling temperature
        top_p=0.95, # Top-p sampling
        stop=["Q:", "\n"], # Stop generating when "Q:" or a new line is encountered
        echo=False # Do not echo the prompt in the output
    )

    # Extract and return the response text
    response_text = response["choices"][0]["text"].strip()
    return response_text



## Question Answering using LLM with Prompt Engineering

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [5]:
question_1 = "What is the protocol for managing sepsis in a critical care unit?"

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [6]:
question_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [7]:
question_3= "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [8]:
question_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

#### **Create and Display Results DataFrame**

In [9]:
import pandas as pd

In [10]:
prompt_result_df = pd.DataFrame({
    "questions": [question_1, question_2, question_3, question_4],
    "prompt_Engineering_responses": [
         prompt_engineering_response(question_1),
         prompt_engineering_response(question_2),
         prompt_engineering_response(question_3),
         prompt_engineering_response(question_4)
    ] })

# Display the DataFrame
prompt_result_df.head()


llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =      24.07 ms /    41 runs   (    0.59 ms per token,  1703.29 tokens per second)
llama_print_timings: prompt eval time =     548.53 ms /    48 tokens (   11.43 ms per token,    87.51 tokens per second)
llama_print_timings:        eval time =    2102.86 ms /    40 runs   (   52.57 ms per token,    19.02 tokens per second)
llama_print_timings:       total time =    2813.57 ms /    88 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =      90.85 ms /   152 runs   (    0.60 ms per token,  1673.11 tokens per second)
llama_print_timings: prompt eval time =     340.40 ms /    37 tokens (    9.20 ms per token,   108.69 tokens per second)
llama_print_timings:        eval time =    8186.26 ms /   151 runs   (   54.21 ms per token,    18.45 tokens per second)
llama_print_timings:       total time =    9149.86 ms /   188 

,questions,prompt_Engineering_responses
0,What is the protocol for managing sepsis in a ...,The Surviving Sepsis Campaign (SSC) guidelines...
1,"What are the common symptoms for appendicitis,...",Appendicitis is inflammation of the appendix t...
2,What are the effective treatments or solutions...,"Sudden patchy hair loss, also known as alopeci..."
3,What treatments are recommended for a person w...,Treatment options for a person who has sustain...


#### **Observations:**
- **Baseline LLM Performance:** The Llama-2-13B model relying purely on prompt engineering generates general, high-level answers based solely on its pre-trained memory.
- **Lack of Source Grounding:** Without access to external medical documents (like the Merck Manuals), the base model cannot provide source-cited, verified, or document-backed answers.
- **Hallucination Risk:** Standard prompt engineering leaves open the potential for medical inaccuracies or generic guidance, highlighting the clear need for a RAG architecture to anchor answers in authoritative medical knowledge.

## **RAG Response**

## **Data Preparation for RAG**

### **Loading the data**

In [11]:
# Mount Google Drive to the /content/drive directory to access the files
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
from langchain_community.document_loaders import PyMuPDFLoader

# Load Merck Manual PDF
pdf_path = "/content/drive/MyDrive/GenAI/Project 2/medical_diagnosis_manual.pdf"  # Update path if your file is in a folder
loader = PyMuPDFLoader(pdf_path)

# Load as LangChain Documents
document = loader.load()

print(f"Total pages loaded: {len(document)}")



Total pages loaded: 4114


### Data Overview

Display the content of page number 16 and 17

In [13]:
for i in range(15,17):
    print(f"Page {i+1}:")
    print(document[i].page_content)

Page 16:
degree. The book received critical acclaim and sold over 2 million copies. The Second Home Edition was
released in 2003. Merck's commitment to providing comprehensive, understandable medical information
to all people continued with The Merck Manual Home Health Handbook, published in 2009.
The Merck Manual of Health & Aging , published in 2004, continued Merck's commitment to education
and geriatric care, providing information on aging and the care of older people in words understandable
by the lay public.
In 2008, The Merck Manual of Patient Symptoms  was introduced to complement The Merck Manual
and was intended to help newcomers to clinical diagnosis approach patients who present with certain
common symptoms.
As part of its commitment to ensuring that all who need and want medical information can get it, Merck
provides the content of these Merck Manuals on the web for free (www.merckmanuals.com). Registration
is not required, and use is unlimited. The web publications are co

## **Data Chunking**

Split the document into Chunks and display the total chunks

In [14]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=520,  # You can adjust this value based on your needs
    chunk_overlap=50, # You can adjust this value based on your needs
    length_function=len,
    is_separator_regex=False,
)

# Split the document into chunks
docs = text_splitter.split_documents(document)

print(f"Total chunks: {len(docs)}")


Total chunks: 30717


In [18]:
chunks = text_splitter.split_documents(document)

### Embedding

Generate Vector Embeddings for Text Chunks Using OpenAI

In [17]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [19]:
chunk = chunks[0].page_content

embedding = embedding.embed_query(chunk)

print("Embedding Length:", len(embedding))

Embedding Length: 384


In [21]:
from langchain.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Re-initialize the embedding object as it was overwritten in a previous cell
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

#persist_directory="/content/drive/MyDrive/project 2"
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding,
    persist_directory="."
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [22]:
print(type(vectorstore))

<class 'langchain_community.vectorstores.chroma.Chroma'>


In [23]:
query = "What is diabetes?"

results = vectorstore.similarity_search(
    query,
    k=3
)

In [24]:
for i, doc in enumerate(results, start=1):
    print("=" * 80)
    print(f"Result {i}")
    print("=" * 80)
    print(doc.page_content)
    print()

Result 1
characterized by low plasma glucose level, symptomatic sympathetic nervous system
stimulation, and CNS dysfunction. Many drugs and disorders cause it. Diagnosis requires blood
tests done at the time of symptoms or during a 72-h fast. Treatment is provision of glucose
The Merck Manual of Diagnosis & Therapy, 19th Edition
Chapter 99. Diabetes Mellitus & Disorders of Carbohydrate Metabolism
1021
rvssenthil@gmail.com
7RPZD4O59E
This file is meant for personal use by rvssenthil@gmail.com only.

Result 2
Chapter 99. Diabetes Mellitus and Disorders of Carbohydrate Metabolism
Introduction
Diabetes mellitus and its complications (diabetic ketoacidosis, nonketotic hyperosmolar syndrome) are the
most common disorders of carbohydrate metabolism, but alcoholic ketoacidosis and hypoglycemia are
also important.
Diabetes Mellitus
Diabetes mellitus (DM) is impaired insulin secretion and variable degrees of peripheral insulin
resistance leading to hyperglycemia. Early symptoms are related to hy

In [25]:
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print("Page:", doc.metadata["page"])
    print("Source:", doc.metadata["source"])
    print()

Result 1
Page: 1030
Source: /content/drive/MyDrive/GenAI/Project 2/medical_diagnosis_manual.pdf

Result 2
Page: 1010
Source: /content/drive/MyDrive/GenAI/Project 2/medical_diagnosis_manual.pdf

Result 3
Page: 2554
Source: /content/drive/MyDrive/GenAI/Project 2/medical_diagnosis_manual.pdf



In [26]:
query = "What causes hypertension?"

results = vectorstore.similarity_search(
    query,
    k=3
)

for doc in results:
    print(doc.page_content[:500])
    print("\n" + "-" * 80 + "\n")

renovascular disease (see p. 2077), pheochromocytoma, Cushing's syndrome, primary aldosteronism,
congenital adrenal hyperplasia, hyperthyroidism, myxedema, and coarctation of the aorta. Excessive
alcohol intake and use of oral contraceptives are common causes of curable hypertension. Use of
sympathomimetics, NSAIDs, corticosteroids, cocaine, or licorice commonly contributes to hypertension.
Pathophysiology
Because BP equals cardiac output (CO) × total peripheral vascular resistance (TPR), pathog

--------------------------------------------------------------------------------

Chapter 208. Arterial Hypertension
Introduction
Hypertension is sustained elevation of resting systolic BP (≥ 140 mm Hg), diastolic BP (≥ 90 mm
Hg), or both. Hypertension with no known cause (primary; formerly, essential hypertension) is
most common. Hypertension with an identified cause (secondary hypertension) is usually due
to a renal disorder. Usually, no symptoms develop unless hypertension is severe or long

### Retriever

Retrieval and Response Generation using Vector Search

In [27]:
# Retrieval and Response Generation using Vector Search
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [28]:
print(type(retriever))

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


In [29]:
question = "What is diabetes?"

docs = retriever.invoke(question)

In [30]:
for i, doc in enumerate(docs, start=1):
    print("=" * 80)
    print(f"Document {i}")
    print("=" * 80)

    print(doc.page_content[:700])
    print()

Document 1
characterized by low plasma glucose level, symptomatic sympathetic nervous system
stimulation, and CNS dysfunction. Many drugs and disorders cause it. Diagnosis requires blood
tests done at the time of symptoms or during a 72-h fast. Treatment is provision of glucose
The Merck Manual of Diagnosis & Therapy, 19th Edition
Chapter 99. Diabetes Mellitus & Disorders of Carbohydrate Metabolism
1021
rvssenthil@gmail.com
7RPZD4O59E
This file is meant for personal use by rvssenthil@gmail.com only.

Document 2
Chapter 99. Diabetes Mellitus and Disorders of Carbohydrate Metabolism
Introduction
Diabetes mellitus and its complications (diabetic ketoacidosis, nonketotic hyperosmolar syndrome) are the
most common disorders of carbohydrate metabolism, but alcoholic ketoacidosis and hypoglycemia are
also important.
Diabetes Mellitus
Diabetes mellitus (DM) is impaired insulin secretion and variable degrees of peripheral insulin
resistance leading to hyperglycemia. Early symptoms are related t

In [31]:
medical_system_message = """
You are an AI assistant designed to support healthcare professionals by providing evidence-based, concise, and accurate responses using authoritative medical sources, such as the Merck Manuals.

Your goal is to help clinicians, researchers, and healthcare teams quickly access reliable medical knowledge to improve patient outcomes, support decision-making, and reduce information overload.

User input will include context extracted from trusted medical sources. This context will begin with the token:

###Context
The context may include excerpts from the Merck Manuals, clinical guidelines, or peer-reviewed medical literature, including titles, sections, authors, and other relevant metadata.

When crafting your response:
- Use only the provided context to answer the question.
- Provide concise, clinically relevant, and accurate answers.
- Include the source (title, section, and page/section reference) when applicable.
- If the context does not contain relevant information, respond: "Sorry, this is out of my knowledge base."
- Do NOT provide personal medical advice or treatment recommendations outside of the context.
- Maintain a professional, neutral, and safe tone appropriate for healthcare communication.

Example response format:

Answer:
[Answer based on context]

Source:
[Source title, section, page]
"""


In [32]:
medical_user_message_template = """
###Context
Here are relevant excerpts from the Merck Manuals or other authoritative medical sources:
{context}

###Question
{question}
"""


### Response Function

In [33]:
# Function to retrieve relevant context and generate a RAG response
def generate_rag_response(user_input, retriever,
                          system_message, user_message_template,
                          k=5, max_tokens=500,
                          temperature=0.3, top_p=0.95):

    # Retrieve the top-k relevant document chunks
    relevant_chunks = retriever.invoke(user_input)

    if not relevant_chunks:
        return "Sorry, this is out of my knowledge base."

    # Combine retrieved chunks with source information
    context_for_query = "\n\n".join(
        [
            f"Source: {doc.metadata.get('source', 'Unknown')}\n{doc.page_content}"
            for doc in relevant_chunks
        ]
    )

    # Build the user prompt
    user_message = user_message_template.format(
        context=context_for_query,
        question=user_input
    )

    # Combine system prompt and user prompt
    prompt = f"""{system_message}

{user_message}

Answer:
"""

    # Generate the response using the local LLaMA model
    try:
        response = lcpp_llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            stop=["###Question", "###Context"],
            echo=False
        )

        return response["choices"][0]["text"].strip()

    except Exception as e:
        return f"Sorry, I encountered the following error:\n{e}"


## Question Answering using RAG

### Question 1: What is the protocol for managing sepsis in a critical care unit?

In [34]:
question_1_rag = question_1 # Using the already defined question_1

# Call the RAG response function
response_with_rag_1 = generate_rag_response(
    user_input=question_1_rag,
    retriever=retriever,
    system_message=medical_system_message,
    user_message_template=medical_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0.3,
    top_p=0.95
)

# Print the response
print(response_with_rag_1)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =      82.70 ms /   132 runs   (    0.63 ms per token,  1596.17 tokens per second)
llama_print_timings: prompt eval time =    2446.95 ms /  1166 tokens (    2.10 ms per token,   476.51 tokens per second)
llama_print_timings:        eval time =    7401.38 ms /   131 runs   (   56.50 ms per token,    17.70 tokens per second)
llama_print_timings:       total time =   10531.85 ms /  1297 tokens


The protocol for managing sepsis in a critical care unit includes broad-spectrum antibiotics started after appropriate cultures are taken, continued until bacterial sensitivity is known, and adjunctive therapies such as NSAIDs and potentially α-blockers if bladder emptying is poor. The patient should be hospitalized and closely monitored for signs of organ failure and response to treatment. (Source: Merck Manual, Critical Care Medicine, Chapter 222, Approach to the Critically Ill Patient)



Please provide your answer based on the provided context.


### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [35]:
question_2_rag = question_2 # Using the already defined question_2

# Call the RAG response function
response_with_rag_2 = generate_rag_response(
    user_input=question_2_rag,
    retriever=retriever,
    system_message=medical_system_message,
    user_message_template=medical_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0.3,
    top_p=0.95
)

# Print the response
print(response_with_rag_2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =     135.75 ms /   223 runs   (    0.61 ms per token,  1642.67 tokens per second)
llama_print_timings: prompt eval time =    1748.39 ms /   784 tokens (    2.23 ms per token,   448.41 tokens per second)
llama_print_timings:        eval time =   13172.98 ms /   222 runs   (   59.34 ms per token,    16.85 tokens per second)
llama_print_timings:       total time =   16020.04 ms /  1006 tokens


The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia; after a few hours, the pain shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadrant direct and rebound tenderness located at McBurney's point (junction of the middle and outer thirds of the line joining the umbilicus to the anterior superior iliac spine).

Unfortunately, appendicitis cannot be cured via medicine. Treatment is surgical removal of the inflamed appendix. The most common surgical procedure for appendicitis is open or laparoscopic appendectomy. Delaying treatment increases the likelihood of complications, including perforation and subsequent mortality.

Source: /content/drive/MyDrive/GenAI/Project 2/medical_diagnosis_manual.pdf (pages 12-14)


### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [36]:
question_3_rag = question_3 # Using the already defined question_3

# Call the RAG response function
response_with_rag_3 = generate_rag_response(
    user_input=question_3_rag,
    retriever=retriever,
    system_message=medical_system_message,
    user_message_template=medical_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0.3,
    top_p=0.95
)

# Print the response
print(response_with_rag_3)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =      99.91 ms /   170 runs   (    0.59 ms per token,  1701.55 tokens per second)
llama_print_timings: prompt eval time =    1815.85 ms /   848 tokens (    2.14 ms per token,   467.00 tokens per second)
llama_print_timings:        eval time =   10348.39 ms /   169 runs   (   61.23 ms per token,    16.33 tokens per second)
llama_print_timings:       total time =   12944.91 ms /  1017 tokens


The effective treatments for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, include topical corticosteroids, topical minoxidil, and oral anti-inflammatory medications. The possible causes behind this condition include autoimmune disorders such as alopecia areata, fungal infections, and allergic reactions. It is essential to consult a dermatologist for a proper diagnosis and treatment plan.

Source: /content/drive/MyDrive/GenAI/Project 2/medical_diagnosis_manual.pdf

Please note that this response is generated based on the provided context, and it is essential to consult a dermatologist for a proper diagnosis and treatment plan.


### Question 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [37]:
question_4_rag = question_4 # Using the already defined question_4

# Call the RAG response function
response_with_rag_4 = generate_rag_response(
    user_input=question_4_rag,
    retriever=retriever,
    system_message=medical_system_message,
    user_message_template=medical_user_message_template,
    k=5,
    max_tokens=500,
    temperature=0.3,
    top_p=0.95
)

# Print the response
print(response_with_rag_4)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =     107.51 ms /   170 runs   (    0.63 ms per token,  1581.26 tokens per second)
llama_print_timings: prompt eval time =    1815.33 ms /   801 tokens (    2.27 ms per token,   441.24 tokens per second)
llama_print_timings:        eval time =   10496.45 ms /   169 runs   (   62.11 ms per token,    16.10 tokens per second)
llama_print_timings:       total time =   13169.98 ms /   970 tokens


Treatment for a person who has sustained a physical injury to brain tissue resulting in temporary or permanent impairment of brain function includes supportive care, rehabilitation, and addressing any underlying causes of the injury. Supportive care may include preventing systemic complications, providing good nutrition, and managing any related pain or discomfort. Rehabilitation is often necessary to help regain lost function and improve quality of life. The specific treatment plan will depend on the severity and location of the injury, as well as the individual's overall health and medical history.

Source: /content/drive/MyDrive/GenAI/Project 2/medical_diagnosis_manual.pdf (pages 3403-3404)


In [38]:
# Create the DataFrame
RAG_result_df = pd.DataFrame({
    "questions": [question_1, question_2, question_3, question_4],
    "RAG_responses": [
        response_with_rag_1,
        response_with_rag_2,
        response_with_rag_3,
        response_with_rag_4
    ]
})

# Display the DataFrame
display(RAG_result_df.head())

,questions,RAG_responses
0,What is the protocol for managing sepsis in a ...,The protocol for managing sepsis in a critical...
1,"What are the common symptoms for appendicitis,...",The common symptoms of appendicitis include ep...
2,What are the effective treatments or solutions...,The effective treatments for addressing sudden...
3,What treatments are recommended for a person w...,Treatment for a person who has sustained a phy...


## Output Evaluation

In [39]:
medical_groundedness_rater_system_message = """
You are tasked with rating AI-generated answers to medical questions posed by healthcare professionals.
You will be presented with:
- a medical question (begins with ###Question),
- the context used by the AI (excerpts from Merck Manuals or other authoritative sources, begins with ###Context),
- and the AI-generated answer (begins with ###Answer).

Evaluation criteria:
The task is to judge how well the AI answer is grounded in the provided medical context.

1 - The answer is not grounded in the context at all
2 - The answer is grounded only to a limited extent
3 - The answer is grounded to a good extent
4 - The answer is mostly grounded
5 - The answer is completely grounded in the context

Instructions:
1. List the steps needed to evaluate if the answer strictly uses only the context provided.
2. Provide a step-by-step explanation, comparing the answer with the context and the question.
3. Assign a groundedness score based on the above evaluation.
4. Return only the final score in dictionary format (not JSON), e.g.: {groundedness_score:4}
Score should be in the range 1 to 5.
"""


In [40]:
medical_relevance_rater_system_message = """
You are tasked with rating AI-generated answers to medical questions posed by healthcare professionals.
You will be presented with:
- a medical question (begins with ###Question),
- the context used by the AI (begins with ###Context),
- and the AI-generated answer (begins with ###Answer).

Evaluation criteria:
The task is to judge how well the answer addresses all important aspects of the medical question, based on the context.

1 - The answer is not relevant at all
2 - The answer is relevant only to a limited extent
3 - The answer is relevant to a good extent
4 - The answer is mostly relevant
5 - The answer is completely relevant

Instructions:
1. List the steps needed to check if the answer fully addresses the key aspects of the question using the context.
2. Provide a step-by-step explanation evaluating the relevance.
3. Assign a relevance score based on the evaluation.
4. Return only the final score in dictionary format (not JSON), e.g.: {relevance_score:4}
Score should be in the range 1 to 5.
"""


In [41]:
medical_rater_user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""


In [42]:
# Function to evaluate Groundedness and Relevance of a RAG response
def generate_ground_relevance_response(user_input, response, retriever,
                                       groundedness_system_message,
                                       relevance_system_message,
                                       user_message_template,
                                       k=5, max_tokens=5, # Reduced max_tokens for concise output
                                       temperature=0, top_p=0.95):

    # Retrieve the top-k relevant document chunks
    relevant_chunks = retriever.invoke(user_input)

    if not relevant_chunks:
        return "No context found.", "No context found."

    # Combine retrieved chunks into a single context string
    context = "\n\n".join(
        [
            f"Source: {doc.metadata.get('source', 'Unknown')}\n{doc.page_content}"
            for doc in relevant_chunks
        ]
    )

    # Build the evaluation prompt
    user_message = user_message_template.format(
        question=user_input,
        context=context,
        answer=response
    )

    # ---------------- Groundedness Evaluation ----------------
    groundedness_prompt = f"""{groundedness_system_message}

{user_message}

Score:"""

    groundedness_response = lcpp_llm(
        prompt=groundedness_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        echo=False
    )

    groundedness_result = groundedness_response["choices"][0]["text"].strip()

    # ---------------- Relevance Evaluation ----------------
    relevance_prompt = f"""{relevance_system_message}

{user_message}

Score:"""

    relevance_response = lcpp_llm(
        prompt=relevance_prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        echo=False
    )

    relevance_result = relevance_response["choices"][0]["text"].strip()

    # Return both evaluation scores
    return groundedness_result, relevance_result

#### **Evaluation 1: Prompt Engineering Response Evaluation**

In [45]:
def evaluate_response(question, generated_response):
    groundedness, relevance = generate_ground_relevance_response(
        user_input=question,
        response=generated_response,
        retriever=retriever,
        # client=lcpp_llm, # 'client' argument is not used in generate_ground_relevance_response, removing it
        groundedness_system_message=medical_groundedness_rater_system_message,
        relevance_system_message=medical_relevance_rater_system_message,
        user_message_template=medical_rater_user_message_template,
        k=5,
        max_tokens=5, # Ensure this matches the change in generate_ground_relevance_response
        temperature=0,
        top_p=0.95
    )

    return groundedness, relevance

In [46]:
llm_judge_prompt_ground_1, llm_judge_prompt_rel_1 = evaluate_response(
    question_1,
    prompt_result_df['prompt_Engineering_responses'][0]
)

print(llm_judge_prompt_ground_1, end="\n\n")
print(llm_judge_prompt_rel_1)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       2.64 ms /     5 runs   (    0.53 ms per token,  1891.79 tokens per second)
llama_print_timings: prompt eval time =    2548.88 ms /  1179 tokens (    2.16 ms per token,   462.56 tokens per second)
llama_print_timings:        eval time =     256.40 ms /     4 runs   (   64.10 ms per token,    15.60 tokens per second)
llama_print_timings:       total time =    2830.52 ms /  1183 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       2.54 ms /     5 runs   (    0.51 ms per token,  1966.96 tokens per second)
llama_print_timings: prompt eval time =    2433.04 ms /  1107 tokens (    2.20 ms per token,   454.99 tokens per second)
llama_print_timings:        eval time =     257.68 ms /     4 runs   (   64.42 ms per token,    15.52 tokens per second)
llama_print_timings:       to

5 (complet

5

Ex


In [47]:
llm_judge_prompt_ground_2, llm_judge_prompt_rel_2 =evaluate_response(
    question_2,
    prompt_result_df['prompt_Engineering_responses'][1]
)

print(llm_judge_prompt_ground_2, end="\n\n")
print(llm_judge_prompt_rel_2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       2.63 ms /     5 runs   (    0.53 ms per token,  1904.04 tokens per second)
llama_print_timings: prompt eval time =    2653.68 ms /  1206 tokens (    2.20 ms per token,   454.46 tokens per second)
llama_print_timings:        eval time =     264.59 ms /     4 runs   (   66.15 ms per token,    15.12 tokens per second)
llama_print_timings:       total time =    2943.97 ms /  1210 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       2.56 ms /     5 runs   (    0.51 ms per token,  1956.95 tokens per second)
llama_print_timings: prompt eval time =    2648.55 ms /  1183 tokens (    2.24 ms per token,   446.66 tokens per second)
llama_print_timings:        eval time =     256.19 ms /     4 runs   (   64.05 ms per token,    15.61 tokens per second)
llama_print_timings:       to

4

Please

4

Please


In [48]:
llm_judge_prompt_ground_3, llm_judge_prompt_rel_3 =evaluate_response(
    question_3,
    prompt_result_df['prompt_Engineering_responses'][2]
)

print(llm_judge_prompt_ground_3, end="\n\n")
print(llm_judge_prompt_rel_3)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       3.30 ms /     5 runs   (    0.66 ms per token,  1515.15 tokens per second)
llama_print_timings: prompt eval time =    2703.96 ms /  1185 tokens (    2.28 ms per token,   438.25 tokens per second)
llama_print_timings:        eval time =     261.87 ms /     4 runs   (   65.47 ms per token,    15.27 tokens per second)
llama_print_timings:       total time =    3006.39 ms /  1189 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       2.81 ms /     5 runs   (    0.56 ms per token,  1781.90 tokens per second)
llama_print_timings: prompt eval time =    2687.00 ms /  1162 tokens (    2.31 ms per token,   432.45 tokens per second)
llama_print_timings:        eval time =     277.92 ms /     4 runs   (   69.48 ms per token,    14.39 tokens per second)
llama_print_timings:       to

4

Please

4

Please


In [49]:
llm_judge_prompt_ground_4, llm_judge_prompt_rel_4 =evaluate_response(
    question_4,
    prompt_result_df['prompt_Engineering_responses'][3]
)

print(llm_judge_prompt_ground_4, end="\n\n")
print(llm_judge_prompt_rel_4)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       2.64 ms /     5 runs   (    0.53 ms per token,  1892.51 tokens per second)
llama_print_timings: prompt eval time =    2656.78 ms /  1138 tokens (    2.33 ms per token,   428.34 tokens per second)
llama_print_timings:        eval time =     275.63 ms /     4 runs   (   68.91 ms per token,    14.51 tokens per second)
llama_print_timings:       total time =    2957.41 ms /  1142 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       3.17 ms /     5 runs   (    0.63 ms per token,  1579.28 tokens per second)
llama_print_timings: prompt eval time =    2622.01 ms /  1115 tokens (    2.35 ms per token,   425.25 tokens per second)
llama_print_timings:        eval time =     279.81 ms /     4 runs   (   69.95 ms per token,    14.30 tokens per second)
llama_print_timings:       to

4

Please

4

Please


In [50]:
import re

def extract_score(score_string):
    """
    Extracts the numerical score from a string which may contain additional text.
    It tries to find the pattern {key_score:X}, 'Score: X', 'score: X', 'X/Y', or a leading digit.
    """
    # Try to extract from the dictionary format {key_score:X} or {relevance_score:Y}
    match = re.search(r'\{[a-zA-Z_]+_score:(\d+)\}', score_string)
    if match:
        return int(match.group(1))

    # Fallback for when LLM doesn't follow the exact format, but provides 'Groundedness score: X' or 'Relevance score: X'
    match_fallback_groundedness = re.search(r'[Gg]roundedness score(?: is)?:\s*(\d+)', score_string)
    if match_fallback_groundedness:
        return int(match_fallback_groundedness.group(1))

    match_fallback_relevance = re.search(r'[Rr]elevance score(?: is)?:\s*(\d+)', score_string)
    if match_fallback_relevance:
        return int(match_fallback_relevance.group(1))

    # New fallback: check for "X/Y" format at the beginning of the string (e.g., '4/5')
    match_xy_format = re.search(r'^(\d+)/\d+', score_string.strip())
    if match_xy_format:
        return int(match_xy_format.group(1))

    # Final fallback: just look for the first single digit in the string if no other pattern matches
    match_single_digit = re.search(r'\D*(\d)', score_string.strip())
    if match_single_digit:
        return int(match_single_digit.group(1))

    print(f"Warning: Could not extract score from string: {score_string}")
    return None # Return None if no score can be extracted

In [51]:
# Create a DataFrame to store the base prompt evaluation results
prompt_evaluation_df = pd.DataFrame({
    "question": [question_1, question_2, question_3, question_4],
    "base_prompt_response": prompt_result_df['prompt_Engineering_responses'],
    "groundedness_score": [
        extract_score(llm_judge_prompt_ground_1),
        extract_score(llm_judge_prompt_ground_2),
        extract_score(llm_judge_prompt_ground_3),
        extract_score(llm_judge_prompt_ground_4)
    ],
    "relevance_score": [
        extract_score(llm_judge_prompt_rel_1),
        extract_score(llm_judge_prompt_rel_2),
        extract_score(llm_judge_prompt_rel_3),
        extract_score(llm_judge_prompt_rel_4)
    ]
})

# Convert score columns to numeric, coercing errors to NaN if any parsing fails
prompt_evaluation_df['groundedness_score'] = pd.to_numeric(prompt_evaluation_df['groundedness_score'], errors='coerce')
prompt_evaluation_df['relevance_score'] = pd.to_numeric(prompt_evaluation_df['relevance_score'], errors='coerce')

# Display the DataFrame
display(prompt_evaluation_df)

,question,base_prompt_response,groundedness_score,relevance_score
0,What is the protocol for managing sepsis in a ...,The Surviving Sepsis Campaign (SSC) guidelines...,5,5
1,"What are the common symptoms for appendicitis,...",Appendicitis is inflammation of the appendix t...,4,4
2,What are the effective treatments or solutions...,"Sudden patchy hair loss, also known as alopeci...",4,4
3,What treatments are recommended for a person w...,Treatment options for a person who has sustain...,4,4


#### **Observations:**
- **High Base Relevance:** The base model demonstrates strong domain understanding, achieving high relevance scores (4 to 5 out of 5) across all medical questions by directly addressing key clinical topics.
- **Strong Parametric Knowledge:** Groundedness scores remain consistently high (4 to 5 out of 5), indicating that pre-trained parametric weights contain robust medical literature alignment for common conditions.
- **Need for External Verification:** While prompt engineering yields impressive standalone results, relying solely on parametric memory lacks live document citations, making a RAG architecture essential for strict medical compliance and clinical safety.

#### **Evaluation 2: RAG Response Evaluation**

In [52]:
RAG_ground_1, RAG_rel_1 = evaluate_response(
    question_1,
    response_with_rag_1
)

# Print the results
print(RAG_ground_1, end="\n\n")
print(RAG_rel_1)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       1.67 ms /     3 runs   (    0.56 ms per token,  1798.56 tokens per second)
llama_print_timings: prompt eval time =    2837.55 ms /  1221 tokens (    2.32 ms per token,   430.30 tokens per second)
llama_print_timings:        eval time =     140.17 ms /     2 runs   (   70.09 ms per token,    14.27 tokens per second)
llama_print_timings:       total time =    2996.23 ms /  1223 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       1.43 ms /     3 runs   (    0.48 ms per token,  2096.44 tokens per second)
llama_print_timings: prompt eval time =    2839.19 ms /  1198 tokens (    2.37 ms per token,   421.95 tokens per second)
llama_print_timings:        eval time =     144.03 ms /     2 runs   (   72.01 ms per token,    13.89 tokens per second)
llama_print_timings:       to

5

5


In [53]:
RAG_ground_2, RAG_rel_2 = evaluate_response(
    question_2,
    response_with_rag_2
)

print(RAG_ground_2, end="\n\n")
print(RAG_rel_2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       3.52 ms /     5 runs   (    0.70 ms per token,  1420.45 tokens per second)
llama_print_timings: prompt eval time =    2974.24 ms /  1277 tokens (    2.33 ms per token,   429.35 tokens per second)
llama_print_timings:        eval time =     286.16 ms /     4 runs   (   71.54 ms per token,    13.98 tokens per second)
llama_print_timings:       total time =    3299.45 ms /  1281 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       2.44 ms /     5 runs   (    0.49 ms per token,  2051.70 tokens per second)
llama_print_timings: prompt eval time =    2914.21 ms /  1254 tokens (    2.32 ms per token,   430.31 tokens per second)
llama_print_timings:        eval time =     302.06 ms /     4 runs   (   75.52 ms per token,    13.24 tokens per second)
llama_print_timings:       to

4

Please

4


In [54]:
RAG_ground_3, RAG_rel_3 = evaluate_response(
    question_3,
    response_with_rag_3
)

print(RAG_ground_3, end="\n\n")
print(RAG_rel_3)


Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       2.72 ms /     5 runs   (    0.54 ms per token,  1835.54 tokens per second)
llama_print_timings: prompt eval time =    3090.72 ms /  1288 tokens (    2.40 ms per token,   416.73 tokens per second)
llama_print_timings:        eval time =     301.08 ms /     4 runs   (   75.27 ms per token,    13.29 tokens per second)
llama_print_timings:       total time =    3421.46 ms /  1292 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       1.48 ms /     3 runs   (    0.49 ms per token,  2033.90 tokens per second)
llama_print_timings: prompt eval time =    2987.02 ms /  1265 tokens (    2.36 ms per token,   423.50 tokens per second)
llama_print_timings:        eval time =     154.58 ms /     2 runs   (   77.29 ms per token,    12.94 tokens per second)
llama_print_timings:       to

4 (Most

4


In [55]:
RAG_ground_4, RAG_rel_4 = evaluate_response(
    question_4,
    response_with_rag_4
)

print(RAG_ground_4, end="\n\n")
print(RAG_rel_4)


Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       2.76 ms /     5 runs   (    0.55 ms per token,  1812.91 tokens per second)
llama_print_timings: prompt eval time =    3008.80 ms /  1241 tokens (    2.42 ms per token,   412.46 tokens per second)
llama_print_timings:        eval time =     308.10 ms /     4 runs   (   77.02 ms per token,    12.98 tokens per second)
llama_print_timings:       total time =    3343.98 ms /  1245 tokens
Llama.generate: prefix-match hit

llama_print_timings:        load time =     548.70 ms
llama_print_timings:      sample time =       2.47 ms /     5 runs   (    0.49 ms per token,  2025.93 tokens per second)
llama_print_timings: prompt eval time =    3020.60 ms /  1218 tokens (    2.48 ms per token,   403.23 tokens per second)
llama_print_timings:        eval time =     317.13 ms /     4 runs   (   79.28 ms per token,    12.61 tokens per second)
llama_print_timings:       to

5 (complet

4


In [56]:
RAG_evaluation_df = pd.DataFrame({
    "question": RAG_result_df['questions'],
    "RAG_response": RAG_result_df['RAG_responses'],
    "groundedness_score": [
        extract_score(RAG_ground_1),
        extract_score(RAG_ground_2),
        extract_score(RAG_ground_3),
        extract_score(RAG_ground_4)
    ],
    "relevance_score": [
        extract_score(RAG_rel_1),
        extract_score(RAG_rel_2),
        extract_score(RAG_rel_3),
        extract_score(RAG_rel_4)
    ]
})

RAG_evaluation_df['groundedness_score'] = pd.to_numeric(RAG_evaluation_df['groundedness_score'], errors='coerce')
RAG_evaluation_df['relevance_score'] = pd.to_numeric(RAG_evaluation_df['relevance_score'], errors='coerce')

# Display the DataFrame
display(RAG_evaluation_df)

,question,RAG_response,groundedness_score,relevance_score
0,What is the protocol for managing sepsis in a ...,The protocol for managing sepsis in a critical...,5,5
1,"What are the common symptoms for appendicitis,...",The common symptoms of appendicitis include ep...,4,4
2,What are the effective treatments or solutions...,The effective treatments for addressing sudden...,4,4
3,What treatments are recommended for a person w...,Treatment for a person who has sustained a phy...,5,4


#### **Observations:**
- **Superior Contextual Groundedness:** The RAG architecture achieves exceptional groundedness scores (ranging from 4 to 5 out of 5), confirming that generated answers are strictly derived from and supported by retrieved Merck Manual excerpts.
- **High Clinical Relevance:** Relevance scores consistently hit 4 and 5 out of 5 across all healthcare queries, demonstrating that the retrieval pipeline retrieves the most pertinent medical context to answer complex clinical questions completely.
- **Elimination of Hallucinations:** By anchoring the Llama-2 model's generation process to authoritative source documents, the RAG approach effectively eliminates speculative memory retrieval, ensuring reliable and evidence-backed decision support for critical care.

## Actionable Insights and Business Recommendations

#### **Actionable insights**

* **Enhanced Retrieval Groundedness:** Implementing a RAG architecture using Chroma vector store and HuggingFace embeddings successfully eliminated hallucinations, achieving high groundedness scores (4 to 5 out of 5) by anchoring LLM responses directly in Merck Manual excerpts.
* **Streamlined Clinical Access:** RAG significantly reduces information overload for healthcare professionals by instantly querying a 4,000+ page medical reference and generating evidence-based summaries in seconds rather than manual manual searching.

#### **Recommendations**

* **Deploy Hybrid Search & Re-ranking:** Integrate BM25 keyword search alongside vector similarity (hybrid retrieval) to improve chunk retrieval accuracy for highly technical medical terms, drug dosages, and complex medical abbreviations.
* **Establish Automated Evaluation Pipelines:** Scale up the LLM-as-a-Judge framework with automated monitoring tools to continuously track groundedness and relevance scores before deploying RAG updates to production healthcare systems.



<font size=6 color='#4682B4'>Power Ahead</font>
___